In [10]:
import csv
from pathlib import Path

def load_questions(csv_path="../eval/questions.csv"):
    rows = []
    with open(csv_path, mode="r", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            pages = [int(p.strip()) for p in r["expected_pages"].split(";") if p.strip()]
            phrases = [p.strip().lower() for p in r["answer_phrases"].split(";") if p.strip()]
            rows.append({
                "id": r["id"],
                "question": r["question"],
                "type": r["type"],
                "tag": r["tag"],
                "doc_id": r["doc_id"].strip(),
                "expected_pages": pages,
                "answer_phrases": phrases,
                "match": r["match"].strip(),
                "notes": r["notes"].strip(),
            })
    return rows

questions = load_questions()
print(f"Loaded {len(questions)} questions.")
for q in questions:
    print(q["id"], q["type"], q["tag"], q["expected_pages"], q["answer_phrases"])

Loaded 40 questions.
q1 in_scope table [20] ['2/100 leaves']
q2 in_scope multi [29] ['3 to 26', '6 to 15']
q3 in_scope plain [34] ['clear water']
q4 in_scope conflict [4, 7] ['25-28 dae', '18-20 das']
q5 in_scope plain [3] ['2% brine']
q6 in_scope plain [5] ['clearfield']
q7 in_scope table [5] ['120:60:40']
q8 in_scope paraphrase [56] ['solarization']
q9 out_of_scope out_of_scope [] []
q10 out_of_scope out_of_scope [] []
q11 in_scope multi [9, 11] ['25 mg a.i.', 'to avoid sweetening']
q12 in_scope paraphrase [8] ['484 million']
q13 in_scope plain [14] ['furore']
q14 in_scope plain [21] ['25 g']
q15 in_scope table [22] ['lime sulphur + mancozeb 75 wp']
q16 in_scope paraphrase [27] ['leather or fabric shoes']
q17 in_scope plain [37] ['12 kg/ha']
q18 in_scope paraphrase [67] ['cucumber']
q19 in_scope multi [4] ['25 cm', '30 dae']
q20 in_scope paraphrase [6] ['blowing on them with the mouth']
q21 in_scope plain [1] ['60%']
q22 in_scope multi [1] ['31.6', '1028']
q23 in_scope table [23] ['l

In [11]:
import chromadb
from chromadb.config import Settings
from langchain_ollama import OllamaEmbeddings

EMBED_MODEL = "nomic-embed-text"
EXPECTED_DIM = 768
EXPECTED_PREFIX = "none"

emb = OllamaEmbeddings(model=EMBED_MODEL)
client = chromadb.PersistentClient(
    path="../data/chroma/mitra",
    settings=Settings(anonymized_telemetry=False)
)
col = client.get_collection("mitra_chunks")

# Guard: stop immediately if collection metadata does not match query settings
meta = col.metadata or {}
assert meta.get("embed_model") == EMBED_MODEL, f"Model mismatch: {meta}"
assert meta.get("dim") == EXPECTED_DIM, f"Dim mismatch: {meta}"
assert meta.get("doc_prefix") == EXPECTED_PREFIX, f"Prefix mismatch: {meta}"

def norm_text(t):
    return " ".join(t.lower().splitlines())

def chunk_matches_phrase(chunk, q, phrase):
    if chunk["doc_id"] != q["doc_id"]:
        return False
    if chunk["page"] not in q["expected_pages"]:
        return False
    return phrase in norm_text(chunk["text"])

def eval_hits_at_k(chunks_k, q):
    if q["match"] == "any":
        return any(
            chunk_matches_phrase(c, q, ph)
            for c in chunks_k
            for ph in q["answer_phrases"]
        )
    elif q["match"] == "all":
        return all(
            any(chunk_matches_phrase(c, q, ph) for c in chunks_k)
            for ph in q["answer_phrases"]
        )
    raise ValueError(f"Unknown match rule: {q['match']}")

def first_hit_rank(chunks, q):
    for r in range(1, len(chunks) + 1):
        if eval_hits_at_k(chunks[:r], q):
            return r
    return None

eval_records = []
misses = []

for q in questions:
    qv = emb.embed_query(q["question"])
    res = col.query(query_embeddings=[qv], n_results=5)
    
    top5 = []
    for cid, dist, m, doc in zip(
        res["ids"][0], res["distances"][0], res["metadatas"][0], res["documents"][0]
    ):
        top5.append({
            "id": cid,
            "doc_id": m["doc_id"],
            "page": int(m["page"]),
            "similarity": 1.0 - float(dist),
            "text": doc,
        })
    
    if q["type"] == "in_scope":
        h1 = eval_hits_at_k(top5[:1], q)
        h3 = eval_hits_at_k(top5[:3], q)
        h5 = eval_hits_at_k(top5[:5], q)
        rank = first_hit_rank(top5, q)
        eval_records.append({
            "id": q["id"],
            "type": q["type"],
            "tag": q["tag"],
            "hit1": h1,
            "hit3": h3,
            "hit5": h5,
            "first_rank": rank,
            "top1_sim": round(top5[0]["similarity"], 4),
            "top5": top5,
        })
        if not h5:
            misses.append((q, top5[:3]))
    else:
        eval_records.append({
            "id": q["id"],
            "type": q["type"],
            "tag": q["tag"],
            "hit1": None,
            "hit3": None,
            "hit5": None,
            "first_rank": None,
            "top1_sim": round(top5[0]["similarity"], 4),
            "top5": top5,
        })

print("ID  | Tag        | Hit@1 | Hit@3 | Hit@5 | FirstRank | Top1_Sim")
print("-" * 65)
for r in eval_records:
    if r["type"] == "in_scope":
        print(f"{r['id']:<3} | {r['tag']:<10} | {str(r['hit1']):<5} | {str(r['hit3']):<5} | {str(r['hit5']):<5} | {str(r['first_rank']):<9} | {r['top1_sim']:.4f}")
    else:
        print(f"{r['id']:<3} | {r['tag']:<10} |  -    |  -    |  -    |  -        | {r['top1_sim']:.4f}")

ID  | Tag        | Hit@1 | Hit@3 | Hit@5 | FirstRank | Top1_Sim
-----------------------------------------------------------------
q1  | table      | False | False | False | None      | 0.8901
q2  | multi      | True  | True  | True  | 1         | 0.8391
q3  | plain      | True  | True  | True  | 1         | 0.8141
q4  | conflict   | False | False | False | None      | 0.7722
q5  | plain      | False | True  | True  | 3         | 0.8067
q6  | plain      | True  | True  | True  | 1         | 0.8585
q7  | table      | False | True  | True  | 2         | 0.7776
q8  | paraphrase | False | True  | True  | 3         | 0.7743
q9  | out_of_scope |  -    |  -    |  -    |  -        | 0.7557
q10 | out_of_scope |  -    |  -    |  -    |  -        | 0.6249
q11 | multi      | False | False | True  | 4         | 0.8540
q12 | paraphrase | False | True  | True  | 2         | 0.7827
q13 | plain      | True  | True  | True  | 1         | 0.8375
q14 | plain      | True  | True  | True  | 1         | 0.760

In [12]:
import time
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

# B2: Silent retrieve() returning a list of dicts
def retrieve(question, k=5, crop=None):
    qv = emb.embed_query(question)
    res = col.query(
        query_embeddings=[qv],
        n_results=k,
        where={"crop": crop} if crop else None,
    )
    chunks_out = []
    for cid, dist, m, doc in zip(
        res["ids"][0], res["distances"][0], res["metadatas"][0], res["documents"][0]
    ):
        chunks_out.append({
            "id": cid,
            "text": doc,
            "doc_id": m["doc_id"],
            "page": int(m["page"]),
            "similarity": round(1.0 - float(dist), 4),
        })
    return chunks_out

# B3: System prompt rules and context formatter
REFUSAL_SENTENCE = "I could not find this in my documents."

SYSTEM_PROMPT = (
    "You are an offline agricultural assistant for the MITRA rover. "
    "Follow these strict rules:\n"
    "1. Answer ONLY from the numbered context passages below.\n"
    "2. Cite the source passage numbers in square brackets like [1] or [2] after every claim.\n"
    "3. If the answer is not contained in the context, reply with EXACTLY this sentence and nothing else: "
    "'I could not find this in my documents.'\n"
    "4. If the context passages disagree or give different numbers/timings, state both versions clearly with their respective citations.\n"
    "5. Never invent or guess any number, herbicide dose, temperature, or date."
)

def format_context(chunks):
    blocks = []
    for idx, c in enumerate(chunks, start=1):
        blocks.append(f"[{idx}] ({c['doc_id']}, page {c['page']})\n{c['text'].strip()}")
    return "\n\n".join(blocks)

In [13]:
# B4: Local generator with temperature=0 for deterministic, low-hallucination output
llm = ChatOllama(model="gemma3:1b", temperature=0)

# B5: Provisional threshold from Part A score distribution (min_in=0.7722, max_out=0.7557).
# Provisional until Stage 5 expands the evaluation set.
PROVISIONAL_THRESHOLD = 0.765

def answer_question(question, k=5, crop=None):
    t0 = time.perf_counter()
    chunks = retrieve(question, k=k, crop=crop)
    t_retrieve = time.perf_counter() - t0

    top_sim = chunks[0]["similarity"] if chunks else 0.0
    sources = [
        {"ref": i + 1, "doc_id": c["doc_id"], "page": c["page"], "similarity": c["similarity"]}
        for i, c in enumerate(chunks)
    ]

    # B5: Pre-LLM threshold gate — refuse immediately without calling the LLM if retrieval is weak
    if top_sim < PROVISIONAL_THRESHOLD:
        return {
            "answer": REFUSAL_SENTENCE,
            "sources": sources,
            "top_similarity": top_sim,
            "gated": True,
            "timings": {"retrieve_s": round(t_retrieve, 3), "generate_s": 0.0},
        }

    # B4: Build messages and invoke ChatOllama
    ctx_str = format_context(chunks)
    user_prompt = f"Context:\n{ctx_str}\n\nQuestion: {question}\n\nAnswer:"

    t1 = time.perf_counter()
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_prompt),
    ])
    t_generate = time.perf_counter() - t1

    return {
        "answer": response.content.strip(),
        "sources": sources,
        "top_similarity": top_sim,
        "gated": False,
        "timings": {"retrieve_s": round(t_retrieve, 3), "generate_s": round(t_generate, 3)},
    }

In [14]:
stage4_results = []

for q in questions:
    print(f"Running {q['id']} ({q['tag']})...")
    res = answer_question(q["question"], k=5)
    stage4_results.append((q, res))
    
    print(f"  top_sim={res['top_similarity']:.4f} | gated={res['gated']} | "
          f"retrieve={res['timings']['retrieve_s']}s | generate={res['timings']['generate_s']}s")
    print(f"  Answer: {res['answer']}\n")
    for s in res["sources"][:3]:
        print(f"    [{s['ref']}] {s['doc_id']} p.{s['page']} (sim={s['similarity']:.4f})")
    print("-" * 70)

# Save full Stage 4 outputs to ../eval/stage4_results.md for manual audit
md_lines = ["# Stage 4 Generation Results (gemma3:1b, k=5, threshold=0.765)\n"]
for q, res in stage4_results:
    md_lines.append(f"## {q['id']} ({q['type']} / {q['tag']})")
    md_lines.append(f"- **Question:** {q['question']}")
    md_lines.append(f"- **Expected:** `{q['doc_id']}` pages `{q['expected_pages']}` phrases `{q['answer_phrases']}`")
    md_lines.append(f"- **Top-1 Similarity:** `{res['top_similarity']:.4f}` (Gated: `{res['gated']}`)")
    md_lines.append(f"- **Timings:** retrieve `{res['timings']['retrieve_s']}s`, generate `{res['timings']['generate_s']}s`")
    md_lines.append(f"- **Top-5 Sources Passed to Prompt:**")
    for s in res["sources"]:
        md_lines.append(f"  - `[{s['ref']}]` `{s['doc_id']}` page `{s['page']}` (`sim={s['similarity']:.4f}`)")
    md_lines.append(f"\n**Model Answer:**\n> {res['answer']}\n")

Path("../eval/stage4_results.md").write_text("\n".join(md_lines), encoding="utf-8")
print("Saved ../eval/stage4_results.md")

Running q1 (table)...
  top_sim=0.8901 | gated=False | retrieve=0.125s | generate=32.653s
  Answer: Threshold Levels
Thresholds are the levels of pest populations at which you should take pest control action if you want to

    [1] potato_disease-pest-management_cpri_2016_manual p.19 (sim=0.8901)
    [2] potato_disease-pest-management_cpri_2016_manual p.19 (sim=0.7718)
    [3] potato_disease-pest-management_cpri_2016_manual p.68 (sim=0.7652)
----------------------------------------------------------------------
Running q2 (multi)...
  top_sim=0.8391 | gated=False | retrieve=0.306s | generate=39.125s
  Answer: The context states that indirect germination generally occurs at temperature ranges of 6 to 15 °C (optimum 12 °C) by means of zoospore production, and that direct germination takes place under warm temperature (a range of 4 to 30 °C optimum being 25 °C).  The context also specifies that >80% is essential for lesions expansion, and >90% is required for spore formation.

    [1] pot

In [15]:
import time
from pathlib import Path
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

PROMPT_VERSION = "v2"
REFUSAL_SENTENCE = "I could not find this in my documents."
PROVISIONAL_THRESHOLD = 0.765

# 5A-5: Prompt v2 with 2-4 complete sentences rule, conflict rule, and 1-shot unrelated example
SYSTEM_PROMPT_V2 = (
    "You are an offline agricultural assistant for the MITRA rover.\n"
    "Follow these strict rules:\n"
    "1. Answer in 2 to 4 complete sentences using ONLY the numbered context passages below.\n"
    "2. Cite the source passage number in square brackets like [1] or [2] at the end of every sentence.\n"
    "3. If the answer is not in the context, reply with EXACTLY this sentence and nothing else:\n"
    "I could not find this in my documents.\n"
    "4. If the passages disagree on a number or timing, state both versions clearly with their citations.\n"
    "5. Never invent or guess any number, dose, temperature, or date.\n\n"
    "--- Example (unrelated topic) ---\n"
    "Context:\n"
    "[1] (pump_manual_2021, page 3)\n"
    "The solar controller requires a 24 V battery bank and shuts down if voltage drops below 21.5 V.\n\n"
    "[2] (pump_manual_2021, page 8)\n"
    "Table 4: Low-voltage cutoff is set to 22.0 V for lithium packs.\n\n"
    "Question: What battery voltage does the solar controller require and when does it shut down?\n"
    "Answer: The solar controller operates on a 24 V battery bank [1]. The text on page 3 states that "
    "the controller shuts down when voltage falls below 21.5 V [1], whereas Table 4 on page 8 lists a "
    "low-voltage cutoff of 22.0 V for lithium battery packs [2].\n"
    "--- End of Example ---"
)

llm = ChatOllama(model="gemma3:1b", temperature=0)

def format_context(chunks):
    blocks = []
    for idx, c in enumerate(chunks, start=1):
        blocks.append(f"[{idx}] ({c['doc_id']}, page {c['page']})\n{c['text'].strip()}")
    return "\n\n".join(blocks)

def format_programmatic_sources(sources):
    lines = ["Retrieved sources: the model was shown these passages:"]
    for s in sources:
        lines.append(f"  [{s['ref']}] {s['doc_id']}, page {s['page']} (similarity={s['similarity']:.4f})")
    return "\n".join(lines)

def answer_question_v2(question, k=5, crop=None):
    t0 = time.perf_counter()
    chunks = retrieve(question, k=k, crop=crop)
    t_retrieve = time.perf_counter() - t0

    top_sim = chunks[0]["similarity"] if chunks else 0.0
    sources = [
        {"ref": i + 1, "doc_id": c["doc_id"], "page": c["page"], "similarity": c["similarity"]}
        for i, c in enumerate(chunks)
    ]

    if top_sim < PROVISIONAL_THRESHOLD:
        return {
            "prompt_version": PROMPT_VERSION,
            "answer": REFUSAL_SENTENCE,
            "sources": sources,
            "sources_block": format_programmatic_sources(sources),
            "top_similarity": top_sim,
            "gated": True,
            "timings": {"retrieve_s": round(t_retrieve, 3), "generate_s": 0.0},
        }

    ctx_str = format_context(chunks)
    # Reinforce rules right after Question so a 1B model does not lose them after 5,000 chars of context
    user_prompt = (
        f"Context:\n{ctx_str}\n\n"
        f"Question: {question}\n"
        "Remember: Write 2 to 4 complete sentences using ONLY the Context above, cite [1], [2] "
        "after every claim, state both versions if sources disagree, or reply with EXACTLY "
        f"'{REFUSAL_SENTENCE}' if the answer is not in the Context.\n"
        "Answer:"
    )

    t1 = time.perf_counter()
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT_V2),
        HumanMessage(content=user_prompt),
    ])
    t_generate = time.perf_counter() - t1

    return {
        "prompt_version": PROMPT_VERSION,
        "answer": response.content.strip(),
        "sources": sources,
        "sources_block": format_programmatic_sources(sources),
        "top_similarity": top_sim,
        "gated": False,
        "timings": {"retrieve_s": round(t_retrieve, 3), "generate_s": round(t_generate, 3)},
    }

In [16]:
stage4_v2_results = []
for q in questions:
    res = answer_question_v2(q["question"], k=5)
    stage4_v2_results.append((q, res))
    print(f"=== {q['id']} ({q['tag']}) | top_sim={res['top_similarity']:.4f} | gen={res['timings']['generate_s']}s ===")
    print(res["answer"])
    print()
    print(res["sources_block"])
    print("-" * 70)

=== q1 (table) | top_sim=0.8901 | gen=38.117s ===
I could not find this in my documents.
The economic threshold level for aphids and whiteflies is zero, as the text states that even a single pest is unreasonably harmful [1].  Monitoring of arthropod pests is done by trapping or by scouting, and the threshold level is zero for pest populations [2].  Furthermore, the text indicates that the threshold level is zero for economic losses up to 20-50% under severe infestation [3].

Retrieved sources: the model was shown these passages:
  [1] potato_disease-pest-management_cpri_2016_manual, page 19 (similarity=0.8901)
  [2] potato_disease-pest-management_cpri_2016_manual, page 19 (similarity=0.7718)
  [3] potato_disease-pest-management_cpri_2016_manual, page 68 (similarity=0.7652)
  [4] potato_disease-pest-management_cpri_2016_manual, page 62 (similarity=0.7592)
  [5] potato_disease-pest-management_cpri_2016_manual, page 62 (similarity=0.7561)
--------------------------------------------------

In [17]:
PROMPT_VERSION = "v3"
REFUSAL_SENTENCE = "I could not find this in my documents."
PROVISIONAL_THRESHOLD = 0.765

SYSTEM_PROMPT_V3 = (
    "You are an offline agricultural assistant for the MITRA rover.\n"
    "Rules:\n"
    "1. Answer the question in 1 to 3 complete sentences using ONLY the relevant passage(s) from the Context.\n"
    "2. Ignore any Context passages that do not help answer the question.\n"
    "3. Cite the passage number in square brackets like [1] or [2] at the end of every sentence.\n"
    "4. If two passages give different numbers or timings for the same thing, state both versions with their citations.\n"
    "5. Never invent numbers, doses, or dates. If none of the passages answer the question, output only: "
    "I could not find this in my documents.\n\n"
    "Example format:\n"
    "The recommended seed rate is 35-40 kg/ha when sowing in rows 20 cm apart [1]. "
    "If a mechanical power weeder is used, rows should be spaced 25 cm apart [1]."
)

def answer_question_v3(question, k=5, crop=None):
    t0 = time.perf_counter()
    chunks = retrieve(question, k=k, crop=crop)
    t_retrieve = time.perf_counter() - t0

    top_sim = chunks[0]["similarity"] if chunks else 0.0
    sources = [
        {"ref": i + 1, "doc_id": c["doc_id"], "page": c["page"], "similarity": c["similarity"]}
        for i, c in enumerate(chunks)
    ]

    if top_sim < PROVISIONAL_THRESHOLD:
        return {
            "prompt_version": PROMPT_VERSION,
            "answer": REFUSAL_SENTENCE,
            "sources": sources,
            "sources_block": format_programmatic_sources(sources),
            "top_similarity": top_sim,
            "gated": True,
            "timings": {"retrieve_s": round(t_retrieve, 3), "generate_s": 0.0},
        }

    ctx_str = format_context(chunks)
    user_prompt = (
        f"Context:\n{ctx_str}\n\n"
        f"Question: {question}\n"
        "Instruction: Write 1 to 3 complete sentences answering the Question using ONLY the relevant "
        "Context passage(s) above, and put [1], [2], etc. at the end of every sentence.\n"
        "Answer:"
    )

    t1 = time.perf_counter()
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT_V3),
        HumanMessage(content=user_prompt),
    ])
    t_generate = time.perf_counter() - t1

    ans = response.content.strip()
    # Clean up if 1B model still prepends the refusal line before a cited answer
    lines = [l.strip() for l in ans.splitlines() if l.strip()]
    if len(lines) > 1 and lines[0] == REFUSAL_SENTENCE:
        ans = " ".join(lines[1:])

    return {
        "prompt_version": PROMPT_VERSION,
        "answer": ans,
        "sources": sources,
        "sources_block": format_programmatic_sources(sources),
        "top_similarity": top_sim,
        "gated": False,
        "timings": {"retrieve_s": round(t_retrieve, 3), "generate_s": round(t_generate, 3)},
    }

In [18]:
stage4_v3_results = []
for q in questions:
    res = answer_question_v3(q["question"], k=5)
    stage4_v3_results.append((q, res))
    print(f"=== {q['id']} ({q['tag']}) | top_sim={res['top_similarity']:.4f} | gen={res['timings']['generate_s']}s ===")
    print(res["answer"])
    print()
    print(res["sources_block"])
    print("-" * 70)

=== q1 (table) | top_sim=0.8901 | gen=40.314s ===
[1] The economic threshold level (ETL) for aphids and whiteflies is zero, as even a single pest in such a situation is unreasonably harmful [1].
[2] In some pest control situations, the threshold level is zero: even a single pest in such a situation is unreasonably harmful [2].
[3] Monitoring of arthropod pests is done by trapping or by scouting, and monitoring of weeds is usually done by visual inspection [3].

Retrieved sources: the model was shown these passages:
  [1] potato_disease-pest-management_cpri_2016_manual, page 19 (similarity=0.8901)
  [2] potato_disease-pest-management_cpri_2016_manual, page 19 (similarity=0.7718)
  [3] potato_disease-pest-management_cpri_2016_manual, page 68 (similarity=0.7652)
  [4] potato_disease-pest-management_cpri_2016_manual, page 62 (similarity=0.7592)
  [5] potato_disease-pest-management_cpri_2016_manual, page 62 (similarity=0.7561)
----------------------------------------------------------------

In [19]:
# === Replace the Prompt v2 cells in 04_generation.ipynb with this cell ===
import time
from pathlib import Path
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

PROMPT_VERSION = "v3"
REFUSAL_SENTENCE = "I could not find this in my documents."
PROVISIONAL_THRESHOLD = 0.765

SYSTEM_PROMPT_V3 = (
    "You are an offline agricultural assistant for the MITRA rover.\n"
    "Rules:\n"
    "1. Answer the question in 1 to 3 complete sentences using ONLY the relevant passage(s) from the Context.\n"
    "2. Ignore any Context passages that do not help answer the question.\n"
    "3. Cite the passage number in square brackets like [1] or [2] at the end of every sentence.\n"
    "4. If two passages give different numbers or timings for the same thing, state both versions with their citations.\n"
    "5. Never invent numbers, doses, or dates. If none of the passages answer the question, output only: "
    "I could not find this in my documents.\n\n"
    "Example format:\n"
    "The recommended seed rate is 35-40 kg/ha when sowing in rows 20 cm apart [1]. "
    "If a mechanical power weeder is used, rows should be spaced 25 cm apart [1]."
)

llm = ChatOllama(model="gemma3:1b", temperature=0)

def format_context(chunks):
    return "\n\n".join(
        f"[{idx}] ({c['doc_id']}, page {c['page']})\n{c['text'].strip()}"
        for idx, c in enumerate(chunks, start=1)
    )

def format_programmatic_sources(sources):
    lines = ["Retrieved sources: the model was shown these passages:"]
    for s in sources:
        lines.append(f"  [{s['ref']}] {s['doc_id']}, page {s['page']} (similarity={s['similarity']:.4f})")
    return "\n".join(lines)

def answer_question_v3(question, k=5, crop=None):
    t0 = time.perf_counter()
    chunks = retrieve(question, k=k, crop=crop)
    t_retrieve = time.perf_counter() - t0

    top_sim = chunks[0]["similarity"] if chunks else 0.0
    sources = [
        {"ref": i + 1, "doc_id": c["doc_id"], "page": c["page"], "similarity": c["similarity"]}
        for i, c in enumerate(chunks)
    ]

    if top_sim < PROVISIONAL_THRESHOLD:
        return {
            "prompt_version": PROMPT_VERSION,
            "answer": REFUSAL_SENTENCE,
            "sources": sources,
            "sources_block": format_programmatic_sources(sources),
            "top_similarity": top_sim,
            "gated": True,
            "timings": {"retrieve_s": round(t_retrieve, 3), "generate_s": 0.0},
        }

    ctx_str = format_context(chunks)
    user_prompt = (
        f"Context:\n{ctx_str}\n\n"
        f"Question: {question}\n"
        "Instruction: Write 1 to 3 complete sentences answering the Question using ONLY the relevant "
        "Context passage(s) above, and put [1], [2], etc. at the end of every sentence.\n"
        "Answer:"
    )

    t1 = time.perf_counter()
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT_V3),
        HumanMessage(content=user_prompt),
    ])
    t_generate = time.perf_counter() - t1

    ans = response.content.strip()
    lines = [l.strip() for l in ans.splitlines() if l.strip()]
    if len(lines) > 1 and REFUSAL_SENTENCE.lower() in lines[0].lower():
        ans = " ".join(lines[1:])

    return {
        "prompt_version": PROMPT_VERSION,
        "answer": ans,
        "sources": sources,
        "sources_block": format_programmatic_sources(sources),
        "top_similarity": top_sim,
        "gated": False,
        "timings": {"retrieve_s": round(t_retrieve, 3), "generate_s": round(t_generate, 3)},
    }

stage4_v3_results = []
for q in questions[:10]:  # Run on the original 10 questions (q1-q10)
    res = answer_question_v3(q["question"], k=5)
    stage4_v3_results.append((q, res))
    print(f"=== {q['id']} ({q['tag']}) | top_sim={res['top_similarity']:.4f} | gen={res['timings']['generate_s']}s ===")
    print(res["answer"])
    print()
    print(res["sources_block"])
    print("-" * 70)

=== q1 (table) | top_sim=0.8901 | gen=34.83s ===
[1] The economic threshold level (ETL) for aphids and whiteflies is zero, as even a single pest in such a situation is unreasonably harmful [1].
[2] In some pest control situations, the threshold level is zero: even a single pest in such a situation is unreasonably harmful [2].
[3] Monitoring of arthropod pests is done by trapping or by scouting, and monitoring of weeds is usually done by visual inspection [3].

Retrieved sources: the model was shown these passages:
  [1] potato_disease-pest-management_cpri_2016_manual, page 19 (similarity=0.8901)
  [2] potato_disease-pest-management_cpri_2016_manual, page 19 (similarity=0.7718)
  [3] potato_disease-pest-management_cpri_2016_manual, page 68 (similarity=0.7652)
  [4] potato_disease-pest-management_cpri_2016_manual, page 62 (similarity=0.7592)
  [5] potato_disease-pest-management_cpri_2016_manual, page 62 (similarity=0.7561)
-----------------------------------------------------------------

In [20]:
# === Paste & Run this self-contained cell in 04_generation.ipynb ===
import csv
import time
from pathlib import Path
import numpy as np
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

# 1. Ensure the 10 Stage-4 benchmark questions (q1..q10) are loaded
with open("../eval/questions.csv", mode="r", encoding="utf-8") as f:
    all_qs = list(csv.DictReader(f))
q1_to_q10 = [r for r in all_qs if r["id"] in {f"q{i}" for i in range(1, 11)}]
q1_to_q10.sort(key=lambda r: int(r["id"][1:]))

# 2. If stage4_v3_results is empty or missing, run gemma3:1b (Prompt v3) now
if not globals().get("stage4_v3_results"):
    print("Running gemma3:1b (Prompt v3) on q1..q10...")
    stage4_v3_results = []
    for q in q1_to_q10:
        res = answer_question_v3(q["question"], k=5)
        stage4_v3_results.append((q, res))
        print(f"[gemma3:1b] {q['id']} ({q['tag']}) | top_sim={res['top_similarity']:.4f} | gen={res['timings']['generate_s']}s")
        print(f"  {res['answer']}\n")

# 3. Save full untruncated Prompt v3 outputs to ../eval/stage4_v3_results.md
v3_lines = ["# Stage 4 Prompt v3 Results (gemma3:1b, temperature=0, k=5)\n"]
for q, res in stage4_v3_results:
    v3_lines.append(f"## {q['id']} ({q['tag']}) | top_sim={res['top_similarity']:.4f} | gen={res['timings']['generate_s']}s")
    v3_lines.append(f"**Question:** {q['question']}\n")
    v3_lines.append(f"**Answer (gemma3:1b):**\n{res['answer']}\n")
    v3_lines.append(f"```text\n{res['sources_block']}\n```\n")

Path("../eval/stage4_v3_results.md").write_text("\n".join(v3_lines), encoding="utf-8")
print("Saved ../eval/stage4_v3_results.md\n" + "=" * 70)

# 4. 5B-4: Run gemma3:4b on the exact same retrieved chunks and compare
llm_4b = ChatOllama(model="gemma3:4b", temperature=0)

ret_times_1b = [res["timings"]["retrieve_s"] for _, res in stage4_v3_results]
gen_times_1b = [res["timings"]["generate_s"] for _, res in stage4_v3_results if not res["gated"]]
gen_times_4b = []
stage5_4b_results = []

comp_lines = ["# Stage 5B-4 Generator Comparison: gemma3:1b vs gemma3:4b (Prompt v3, k=5)\n"]

for q, res_1b in stage4_v3_results:
    if res_1b["gated"]:
        stage5_4b_results.append((q, {"answer": REFUSAL_SENTENCE, "gated": True, "generate_s": 0.0}))
        comp_lines.append(f"## {q['id']} ({q['tag']}) — GATED (top_sim={res_1b['top_similarity']:.4f})")
        comp_lines.append(f"- **gemma3:1b (0.0s):** {REFUSAL_SENTENCE}")
        comp_lines.append(f"- **gemma3:4b (0.0s):** {REFUSAL_SENTENCE}\n")
        continue

    chunks = retrieve(q["question"], k=5)
    ctx_str = format_context(chunks)
    user_prompt = (
        f"Context:\n{ctx_str}\n\n"
        f"Question: {q['question']}\n"
        "Instruction: Write 1 to 3 complete sentences answering the Question using ONLY the relevant "
        "Context passage(s) above, and put [1], [2], etc. at the end of every sentence.\n"
        "Answer:"
    )
    t1 = time.perf_counter()
    resp_4b = llm_4b.invoke([
        SystemMessage(content=SYSTEM_PROMPT_V3),
        HumanMessage(content=user_prompt),
    ])
    t_gen = round(time.perf_counter() - t1, 3)
    gen_times_4b.append(t_gen)

    ans_4b = resp_4b.content.strip()
    lines_4b = [l.strip() for l in ans_4b.splitlines() if l.strip()]
    if len(lines_4b) > 1 and REFUSAL_SENTENCE.lower() in lines_4b[0].lower():
        ans_4b = " ".join(lines_4b[1:])

    stage5_4b_results.append((q, {"answer": ans_4b, "gated": False, "generate_s": t_gen}))
    print(f"=== {q['id']} ({q['tag']}) ===")
    print(f"  [1B ({res_1b['timings']['generate_s']}s)]: {res_1b['answer']}")
    print(f"  [4B ({t_gen}s)]: {ans_4b}\n")

    comp_lines.append(f"## {q['id']} ({q['tag']}) | top_sim={res_1b['top_similarity']:.4f}")
    comp_lines.append(f"**Question:** {q['question']}\n")
    comp_lines.append(f"- **gemma3:1b ({res_1b['timings']['generate_s']}s):** {res_1b['answer']}")
    comp_lines.append(f"- **gemma3:4b ({t_gen}s):** {ans_4b}\n")
    comp_lines.append(f"```text\n{res_1b['sources_block']}\n```\n")

p50_ret, p95_ret = np.percentile(ret_times_1b, [50, 95])
p50_1b, p95_1b = np.percentile(gen_times_1b, [50, 95])
p50_4b, p95_4b = np.percentile(gen_times_4b, [50, 95])

summary_block = (
    "\n## Latency Summary (Windows 10 CPU, 12 GB RAM)\n"
    f"- **Retrieval (`nomic-embed-text`):** p50 = `{p50_ret:.3f}s` | p95 = `{p95_ret:.3f}s`\n"
    f"- **Generation (`gemma3:1b`):** p50 = `{p50_1b:.2f}s` | p95 = `{p95_1b:.2f}s`\n"
    f"- **Generation (`gemma3:4b`):** p50 = `{p50_4b:.2f}s` | p95 = `{p95_4b:.2f}s`\n"
)
comp_lines.append(summary_block)
Path("../eval/stage5_4b_comparison.md").write_text("\n".join(comp_lines), encoding="utf-8")
print(summary_block)
print("Saved ../eval/stage5_4b_comparison.md")

Saved ../eval/stage4_v3_results.md
=== q1 (table) ===
  [1B (34.83s)]: [1] The economic threshold level (ETL) for aphids and whiteflies is zero, as even a single pest in such a situation is unreasonably harmful [1].
[2] In some pest control situations, the threshold level is zero: even a single pest in such a situation is unreasonably harmful [2].
[3] Monitoring of arthropod pests is done by trapping or by scouting, and monitoring of weeds is usually done by visual inspection [3].
  [4B (133.594s)]: Economic threshold levels (ETLs) for common potato pests are given in Table 1 [1]. Monitoring is not necessary in situations where a pest is continually present and the threshold is zero, such as for viral diseases in seed potatoes [1]. The morphological identification and monitoring the population of insect vectors, including aphids and whiteflies, is crucial for healthy potato production [4].

=== q2 (multi) ===
  [1B (7.507s)]: The sporangia geminate by two way process i.e. indirect and 